In [10]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

import pandas as pd
import numpy as np

images = pd.read_csv('../../data/images.csv', on_bad_lines='skip').fillna('')
images['id'] = images.filename.str.extract(r'(\d+)').astype(int)

styles = pd.read_csv('../../data/styles.csv', on_bad_lines='skip').fillna('')
styles['context'] = styles.productDisplayName
# styles['context'] += ' #' + styles.gender
# styles['context'] += ' #' + styles.masterCategory
# styles['context'] += ' #' + styles.subCategory
# styles['context'] += ' #' + styles.articleType
# styles['context'] += ' #' + styles.baseColour
# styles['context'] += ' #' + styles.season
# styles['context'] += ' #' + styles.usage

In [11]:
X_train, y_train = styles.context.values, images.link.values
X_test, y_test = [], []

testing = list(zip(X_train, y_train))
np.random.shuffle(testing)

for X, y in testing:
    original = X.split(' ')
    terms = original.copy()
    np.random.shuffle(terms)
    keepers = terms[:-2]
    keep = ' '.join([text for text in original if text in keepers and '#' not in text])
    X_test.append(keep)
    y_test.append(y)
    if len(X_test) >= len(X_train) * 0.2:
        break

X_test = np.array(X_test)
y_test = np.array(y_test)

In [13]:
import sys, os
path_to_root = os.path.abspath("../../")

if path_to_root not in sys.path:
    sys.path.append(path_to_root)

from models.recommender import Recommender

model = Pipeline(
  steps=[
    ('vectorizer', TfidfVectorizer(stop_words='english')),
    ('recommender', Recommender(n_neighbors=20, metric='cosine'))
  ]
)

model.fit(X_train, y_train)
model.score(X_test, y_test, k=20)


np.float64(0.4829262538540207)

In [14]:
model.predict(["UCB Men's Basic Navy Pink T-shirt Blue"])

array([['http://assets.myntassets.com/v1/images/style/properties/a74d995bbda44d66766b70c475b5fca1_images.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/51046a1be1fdbafd655a0f2dae93c50a_images.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/c8bd2106fa00fdf9c9c5e5b2d8bffe6a_images.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/333f214e17fa4e84b98183c02eabd6c5_images.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/c672a8b7de7a87c8812ec589d114d266_images.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/ea96482b320b47a34c9b25157130a4a4_images.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/9728e7ce5a7caf3f06c0ff012c3af63a_images.jpg',
        'http://assets.myntassets.com/assets/images/57679/2018/1/24/11516775023932-NA-6391516775023946-1.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/59da8c38daa82e96c95e17478553160d_images.jpg',
 

In [15]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from gensim.models import Word2Vec

class Word2VecTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=1):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.model = None

    def fit(self, X: np.ndarray, y: np.ndarray = None):
        sentences = [sentence.lower().split() for sentence in X]
        self.model = Word2Vec(
            sentences, 
            vector_size=self.vector_size, 
            window=self.window, 
            min_count=self.min_count
        )
        return self

    def transform(self, X: np.ndarray):
        return np.array([self._get_mean_vector(s) for s in X])

    def _get_mean_vector(self, sentence):
        words = sentence.lower().split()
        vectors = [self.model.wv[w] for w in words if w in self.model.wv]
        
        if not vectors:
            return np.zeros(self.vector_size)
            
        return np.mean(vectors, axis=0)

In [16]:
from sklearn.pipeline import Pipeline
from models.recommender import Recommender

# Build the pipeline
model = Pipeline([
    ('w2v', Word2VecTransformer(vector_size=50, window=3, min_count=1)),
    ('knn', Recommender(n_neighbors=20, metric='cosine'))
])

model.fit(X_train, y=y_train)

# Search for a product
model.score(X_test, y_test, k=50)

np.float64(0.1551995065727593)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm

class BertTransformer(nn.Module, BaseEstimator, TransformerMixin):
    def __init__(
        self,
        embed_dim: int = 64,
        num_heads: int = 4,
        num_layers: int = 2,
        max_len: int = 15,
        epochs: int = 5,
        lr: float = 0.001,
        batch_size: int = 32,
        device: str = "cpu",
        patience: int = 3,
        min_delta: float = 0.001
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_len = max_len
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.patience = patience
        self.min_delta = min_delta
        
        self.word_to_idx = {"<PAD>": 0, "<UNK>": 1}
        self.vocab_size = 2
        
        self.token_embedding = None
        self.pos_embedding = None
        self.transformer = None
        self.to_vocab = None
        self.device = torch.device(device)

    def _tokenize_and_pad(self, X):
        encoded = []
        for text in X:
            tokens = str(text).lower().split()
            ids = [self.word_to_idx.get(w, 1) for w in tokens]
            padded = ids[:self.max_len] + [0] * (self.max_len - len(ids))
            encoded.append(padded)
        return torch.tensor(encoded, dtype=torch.long)

    def _build_vocab(self, X):
        unique_words = set(" ".join(map(str, X)).lower().split())
        for word in unique_words:
            if word not in self.word_to_idx:
                self.word_to_idx[word] = self.vocab_size
                self.vocab_size += 1

    def _init_layers(self):
        self.token_embedding = nn.Embedding(self.vocab_size, self.embed_dim)
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.max_len, self.embed_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim, 
            nhead=self.num_heads, 
            dim_feedforward=self.embed_dim * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers)
        self.to_vocab = nn.Linear(self.embed_dim, self.vocab_size)
        self.to(self.device)

    def forward(self, x):
        x = self.token_embedding(x) + self.pos_embedding[:, :x.size(1), :]
        x = self.transformer(x)
        return x

    def fit(self, X: np.ndarray, y=None):
        self._build_vocab(X)
        self._init_layers()
        
        X_tensor = self._tokenize_and_pad(X)
        dataset = torch.utils.data.TensorDataset(X_tensor)
        loader = torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        optimizer = optim.Adam(self.parameters(), lr=self.lr)
        criterion = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler(device_type=self.device.type, enabled=(self.device.type != "cpu"))

        best_loss = float('inf')
        wait = 0
        
        self.train()
        progress = tqdm(range(self.epochs), desc="Training Bert")

        for epoch in progress:
            total_loss = 0
            batch_progress = tqdm(loader, desc=f"Epoch {epoch + 1}", leave=False)
            
            for [batch] in batch_progress:
                batch = batch.to(self.device)
                optimizer.zero_grad()
                
                with torch.amp.autocast(device_type=self.device.type):
                    hidden = self.forward(batch)
                    logits = self.to_vocab(hidden) 
                    loss = criterion(logits.view(-1, self.vocab_size), batch.view(-1))
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
                batch_progress.set_postfix({"batch_loss": f"{loss.item():.4f}"})
                
            avg_loss = total_loss / len(loader)
            progress.set_postfix({"loss": f"{avg_loss:.4f}", "wait": wait})

            if avg_loss < best_loss - self.min_delta:
                best_loss = avg_loss
                wait = 0
            else:
                wait += 1
                if wait >= self.patience:
                    print(f"\nEarly stopping at epoch {epoch + 1}")
                    break
            
        return self
    
    def transform(self, X: np.ndarray) -> np.ndarray:
        self.eval()
        X_tensor = self._tokenize_and_pad(X)
        loader = torch.utils.data.DataLoader(X_tensor, batch_size=self.batch_size)
        
        embeddings = []
        with torch.inference_mode():
            for batch in loader:
                batch = batch.to(self.device)
                with torch.amp.autocast(device_type=self.device.type):
                    hidden = self.forward(batch)
                    pooled = torch.mean(hidden, dim=1)
                embeddings.append(pooled.cpu().numpy())
        
        return np.vstack(embeddings)

In [35]:
from sklearn.pipeline import Pipeline
from models.recommender import Recommender

# Build the pipeline
model = Pipeline([
    ('bert', BertTransformer(embed_dim=400, epochs=10, device="mps")),
    ('knn', Recommender(n_neighbors=20, metric='cosine'))
])

# Fit and search
model.fit(X_train, y=y_train)
model.score(X_test, y_test, k=5)

Training Bert: 100%|██████████| 10/10 [14:36<00:00, 87.60s/it, loss=0.0000]


np.float64(0.31413094537651187)

In [34]:
model.score(X_test, y_test, k=20)

Batches: 100%|██████████| 278/278 [00:06<00:00, 44.97it/s]


np.float64(0.4621721539050082)

In [33]:
model.named_steps.bert.transform(['United Colors of Benetton Men Black Shoes Shoes']).shape

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]


(1, 384)

In [30]:
from sentence_transformers import SentenceTransformer

class OptimizedBertTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='all-MiniLM-L6-v2', device='cpu'):
        self.model_name = model_name
        self.device = device
        # Load a model already trained for retrieval
        self.model = SentenceTransformer(model_name, device=device)

    def fit(self, X, y=None):
        return self # No training needed! It's already smart.

    def transform(self, X):
        # This will return high-quality embeddings immediately
        return self.model.encode(X, show_progress_bar=True)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
from sklearn.pipeline import Pipeline
from models.recommender import Recommender

# Build the pipeline
model = Pipeline([
    ('bert', OptimizedBertTransformer(device="mps")),
    ('knn', Recommender(n_neighbors=20, metric='cosine'))
])

# Fit and search
model.fit(X_train, y=y_train)
model.score(X_test, y_test, k=5)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1319.99it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 278/278 [00:05<00:00, 52.32it/s]


np.float64(0.4621721539050082)